# SEMMA Framework: Bank Marketing Campaign ResponseDataset: [Bank Marketing - Kaggle](https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset)We apply the **SEMMA** methodology to predict term deposit subscription responses.

## SEMMA Phases- Sample- Explore- Modify- Model- Assess

## SampleSelect a representative subset for agile experimentation while preserving campaign response rates.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

raw = Path('../data/raw/bank-additional-full.csv')
if not raw.exists():
    raise FileNotFoundError('Download bank-additional-full.csv into data/raw before execution.')

bank = pd.read_csv(raw, sep=';')

print(f'Full Dataset Shape: {bank.shape}')
print(f'\nColumns: {list(bank.columns)}')
print(f'\nDataset Info:')
print(bank.info())

# Split into train and holdout (test)
train, holdout = train_test_split(bank, test_size=0.2, stratify=bank['y'], random_state=42)

# SEMMA Sample phase: Take representative sample for rapid prototyping
# In production SEMMA, sampling balances speed vs accuracy
train_sample = train.sample(frac=0.5, random_state=42)  # 50% sample for faster iteration

print(f'\n=== Sampling Strategy ===')
print(f'Full Training Data: {len(train):,} rows')
print(f'Sample Size: {len(train_sample):,} rows ({len(train_sample)/len(train)*100:.1f}%)')
print(f'Holdout (Test) Size: {len(holdout):,} rows')

# Verify class distribution is preserved
print(f'\nClass Distribution in Full Train: {train["y"].value_counts(normalize=True).to_dict()}')
print(f'Class Distribution in Sample: {train_sample["y"].value_counts(normalize=True).to_dict()}')

train_sample.head()

## ExploreProfile the sample, inspect class balance, and produce visuals for the Medium article.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Response rate analysis
response_counts = train_sample['y'].value_counts()
response_rate = train_sample['y'].value_counts(normalize=True)

print('=== Response Rate Analysis ===')
print(f'No subscriptions: {response_counts["no"]:,} ({response_rate["no"]*100:.2f}%)')
print(f'Yes subscriptions: {response_counts["yes"]:,} ({response_rate["yes"]*100:.2f}%)')
print(f'Imbalance Ratio: {response_counts["no"]/response_counts["yes"]:.1f}:1')

# Basic statistics
print(f'\n=== Numeric Features Summary ===')
print(train_sample.describe())

# Visualization
fig, ax = plt.subplots(2, 2, figsize=(14, 10))

# Response distribution
ax[0, 0].bar(['No', 'Yes'], response_counts.values, color=['#3498db', '#2ecc71'], alpha=0.7, edgecolor='black')
ax[0, 0].set_ylabel('Count')
ax[0, 0].set_title('Campaign Response Distribution', fontsize=12, fontweight='bold')
ax[0, 0].grid(axis='y', alpha=0.3)
for i, count in enumerate(response_counts.values):
    ax[0, 0].text(i, count, f'{count:,}', ha='center', va='bottom', fontweight='bold')

# Response rate by job
job_response = train_sample.groupby('job')['y'].apply(lambda x: (x == 'yes').mean() * 100).sort_values(ascending=False)
job_response.plot(kind='barh', ax=ax[0, 1], color='#e74c3c', alpha=0.7, edgecolor='black')
ax[0, 1].set_xlabel('Subscription Rate (%)')
ax[0, 1].set_title('Response Rate by Job Category', fontsize=12, fontweight='bold')
ax[0, 1].grid(axis='x', alpha=0.3)

# Age distribution by response
ax[1, 0].hist([train_sample[train_sample['y']=='no']['age'], 
               train_sample[train_sample['y']=='yes']['age']], 
              bins=30, label=['No', 'Yes'], color=['#3498db', '#2ecc71'], alpha=0.6)
ax[1, 0].set_xlabel('Age')
ax[1, 0].set_ylabel('Frequency')
ax[1, 0].set_title('Age Distribution by Response', fontsize=12, fontweight='bold')
ax[1, 0].legend()
ax[1, 0].grid(alpha=0.3)

# Response rate by education
edu_response = train_sample.groupby('education')['y'].apply(lambda x: (x == 'yes').mean() * 100).sort_values(ascending=False)
edu_response.plot(kind='bar', ax=ax[1, 1], color='#9b59b6', alpha=0.7, edgecolor='black')
ax[1, 1].set_xlabel('Education Level')
ax[1, 1].set_ylabel('Subscription Rate (%)')
ax[1, 1].set_title('Response Rate by Education', fontsize=12, fontweight='bold')
ax[1, 1].tick_params(axis='x', rotation=45)
ax[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Deep dive into key features
fig, ax = plt.subplots(2, 3, figsize=(16, 10))

# Duration vs Response
sns.boxplot(data=train_sample, x='y', y='duration', ax=ax[0, 0])
ax[0, 0].set_title('Call Duration vs Response', fontweight='bold')
ax[0, 0].set_ylabel('Duration (seconds)')

# Campaign contacts vs Response
sns.boxplot(data=train_sample, x='y', y='campaign', ax=ax[0, 1])
ax[0, 1].set_title('Number of Contacts vs Response', fontweight='bold')
ax[0, 1].set_ylabel('Campaign Contacts')

# Previous outcome impact
prev_outcome_rate = train_sample.groupby('poutcome')['y'].apply(lambda x: (x == 'yes').mean() * 100)
prev_outcome_rate.plot(kind='bar', ax=ax[0, 2], color='#f39c12', alpha=0.7, edgecolor='black')
ax[0, 2].set_title('Response Rate by Previous Outcome', fontweight='bold')
ax[0, 2].set_ylabel('Subscription Rate (%)')
ax[0, 2].tick_params(axis='x', rotation=45)
ax[0, 2].grid(axis='y', alpha=0.3)

# Marital status vs Response
marital_response = train_sample.groupby('marital')['y'].apply(lambda x: (x == 'yes').mean() * 100)
marital_response.plot(kind='bar', ax=ax[1, 0], color='#e74c3c', alpha=0.7, edgecolor='black')
ax[1, 0].set_title('Response Rate by Marital Status', fontweight='bold')
ax[1, 0].set_ylabel('Subscription Rate (%)')
ax[1, 0].tick_params(axis='x', rotation=45)
ax[1, 0].grid(axis='y', alpha=0.3)

# Contact type impact
contact_response = train_sample.groupby('contact')['y'].apply(lambda x: (x == 'yes').mean() * 100)
contact_response.plot(kind='bar', ax=ax[1, 1], color='#3498db', alpha=0.7, edgecolor='black')
ax[1, 1].set_title('Response Rate by Contact Type', fontweight='bold')
ax[1, 1].set_ylabel('Subscription Rate (%)')
ax[1, 1].tick_params(axis='x', rotation=0)
ax[1, 1].grid(axis='y', alpha=0.3)

# Month seasonality
month_response = train_sample.groupby('month')['y'].apply(lambda x: (x == 'yes').mean() * 100)
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
month_response = month_response.reindex([m for m in month_order if m in month_response.index])
month_response.plot(kind='line', ax=ax[1, 2], marker='o', color='#2ecc71', linewidth=2, markersize=8)
ax[1, 2].set_title('Seasonality: Response Rate by Month', fontweight='bold')
ax[1, 2].set_ylabel('Subscription Rate (%)')
ax[1, 2].set_xlabel('Month')
ax[1, 2].tick_params(axis='x', rotation=45)
ax[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Key insights
print('\n=== Key Exploration Insights ===')
print(f'Average call duration for Yes: {train_sample[train_sample["y"]=="yes"]["duration"].mean():.1f} sec')
print(f'Average call duration for No: {train_sample[train_sample["y"]=="no"]["duration"].mean():.1f} sec')
print(f'Most responsive job: {job_response.idxmax()} ({job_response.max():.1f}%)')
print(f'Least responsive job: {job_response.idxmin()} ({job_response.min():.1f}%)')

## ModifyFeature engineering and data preparation to boost signal.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Feature Engineering (Modify phase in SEMMA)
print('=== SEMMA Modify Phase: Feature Engineering ===\n')

# Create engineered features
def engineer_features(df):
    df = df.copy()
    
    # Age groups
    df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 50, 65, 100], 
                              labels=['young', 'adult', 'middle', 'senior', 'elderly'])
    
    # Contact intensity
    df['contact_intensity'] = df['campaign'] + df['previous']
    
    # Economic indicator
    df['economic_score'] = df['emp.var.rate'] + df['cons.price.idx'] / 100
    
    # Has previous contact
    df['has_previous'] = (df['previous'] > 0).astype(int)
    
    # Successful previous outcome
    df['prev_success'] = (df['poutcome'] == 'success').astype(int)
    
    return df

train_sample_eng = engineer_features(train_sample)
print('Engineered Features:')
print('- age_group: Categorized age into life stages')
print('- contact_intensity: Total contacts across campaigns')
print('- economic_score: Combined economic indicators')
print('- has_previous: Binary indicator of prior contact')
print('- prev_success: Binary indicator of previous success')

# Prepare features
X = train_sample_eng.drop(columns=['y'])
y = train_sample_eng['y'].map({'no': 0, 'yes': 1})

# Update column lists with engineered features
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

print(f'\nTotal Features: {len(categorical_cols)} categorical + {len(numeric_cols)} numeric = {len(categorical_cols) + len(numeric_cols)}')

# Create preprocessing pipeline
semma_preprocess = ColumnTransformer(transformers=[
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_cols),
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), numeric_cols)
], remainder='drop')

print('\n✓ Preprocessing pipeline created')
print('  - Categorical: Imputation → One-Hot Encoding')
print('  - Numeric: Imputation → Standardization')

## ModelIterate from baseline logistic regression to automated machine learning (e.g., AutoGluon).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

print('=== SEMMA Model Phase: Comparative Modeling ===\n')

# Prepare holdout data with engineered features
holdout_eng = engineer_features(holdout)
X_test = holdout_eng.drop(columns=['y'])
y_test = holdout_eng['y'].map({'no': 0, 'yes': 1})

# Dictionary to store models and their results
models = {}
results = []

# 1. Logistic Regression (interpretable baseline)
print('Training Logistic Regression...')
lr_model = Pipeline([
    ('prep', semma_preprocess), 
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
lr_model.fit(X, y)
lr_probs = lr_model.predict_proba(X_test)[:, 1]
lr_preds = (lr_probs >= 0.3).astype(int)
models['Logistic Regression'] = lr_model
results.append({
    'Model': 'Logistic Regression',
    'Accuracy': (lr_preds == y_test).mean(),
    'Precision': precision_score(y_test, lr_preds),
    'Recall': recall_score(y_test, lr_preds),
    'F1-Score': f1_score(y_test, lr_preds),
    'ROC-AUC': roc_auc_score(y_test, lr_probs)
})
print(f'✓ Logistic Regression trained. ROC-AUC: {results[-1]["ROC-AUC"]:.4f}')

# 2. Decision Tree
print('Training Decision Tree...')
dt_model = Pipeline([
    ('prep', semma_preprocess),
    ('clf', DecisionTreeClassifier(max_depth=10, min_samples_split=50, class_weight='balanced', random_state=42))
])
dt_model.fit(X, y)
dt_probs = dt_model.predict_proba(X_test)[:, 1]
dt_preds = (dt_probs >= 0.3).astype(int)
models['Decision Tree'] = dt_model
results.append({
    'Model': 'Decision Tree',
    'Accuracy': (dt_preds == y_test).mean(),
    'Precision': precision_score(y_test, dt_preds),
    'Recall': recall_score(y_test, dt_preds),
    'F1-Score': f1_score(y_test, dt_preds),
    'ROC-AUC': roc_auc_score(y_test, dt_probs)
})
print(f'✓ Decision Tree trained. ROC-AUC: {results[-1]["ROC-AUC"]:.4f}')

# 3. Random Forest
print('Training Random Forest...')
rf_model = Pipeline([
    ('prep', semma_preprocess),
    ('clf', RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1))
])
rf_model.fit(X, y)
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds = (rf_probs >= 0.3).astype(int)
models['Random Forest'] = rf_model
results.append({
    'Model': 'Random Forest',
    'Accuracy': (rf_preds == y_test).mean(),
    'Precision': precision_score(y_test, rf_preds),
    'Recall': recall_score(y_test, rf_preds),
    'F1-Score': f1_score(y_test, rf_preds),
    'ROC-AUC': roc_auc_score(y_test, rf_probs)
})
print(f'✓ Random Forest trained. ROC-AUC: {results[-1]["ROC-AUC"]:.4f}')

# 4. Gradient Boosting
print('Training Gradient Boosting...')
gb_model = Pipeline([
    ('prep', semma_preprocess),
    ('clf', GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42))
])
gb_model.fit(X, y)
gb_probs = gb_model.predict_proba(X_test)[:, 1]
gb_preds = (gb_probs >= 0.3).astype(int)
models['Gradient Boosting'] = gb_model
results.append({
    'Model': 'Gradient Boosting',
    'Accuracy': (gb_preds == y_test).mean(),
    'Precision': precision_score(y_test, gb_preds),
    'Recall': recall_score(y_test, gb_preds),
    'F1-Score': f1_score(y_test, gb_preds),
    'ROC-AUC': roc_auc_score(y_test, gb_probs)
})
print(f'✓ Gradient Boosting trained. ROC-AUC: {results[-1]["ROC-AUC"]:.4f}')

# 5. XGBoost
print('Training XGBoost...')
scale_pos = (y == 0).sum() / (y == 1).sum()
xgb_model = Pipeline([
    ('prep', semma_preprocess),
    ('clf', XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, scale_pos_weight=scale_pos, random_state=42, eval_metric='logloss'))
])
xgb_model.fit(X, y)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_preds = (xgb_probs >= 0.3).astype(int)
models['XGBoost'] = xgb_model
results.append({
    'Model': 'XGBoost',
    'Accuracy': (xgb_preds == y_test).mean(),
    'Precision': precision_score(y_test, xgb_preds),
    'Recall': recall_score(y_test, xgb_preds),
    'F1-Score': f1_score(y_test, xgb_preds),
    'ROC-AUC': roc_auc_score(y_test, xgb_probs)
})
print(f'✓ XGBoost trained. ROC-AUC: {results[-1]["ROC-AUC"]:.4f}')

print('\n=== Model Comparison Summary ===')
comparison_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print(comparison_df.to_string(index=False))

## AssessSummarize results for marketing stakeholders and document next experiments.

In [ ]:
# SEMMA Assess Phase: Comprehensive Model Assessment
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay

print('=== SEMMA Assess Phase: Model Evaluation & Selection ===\n')

# Select champion model (best ROC-AUC)
champion_name = comparison_df.iloc[0]['Model']
champion_model = models[champion_name]
print(f'🏆 Champion Model: {champion_name}')
print(f'   ROC-AUC: {comparison_df.iloc[0]["ROC-AUC"]:.4f}')
print(f'   F1-Score: {comparison_df.iloc[0]["F1-Score"]:.4f}')

# Get champion predictions
champion_probs = champion_model.predict_proba(X_test)[:, 1]
champion_preds = (champion_probs >= 0.3).astype(int)

# Detailed classification report
print(f'\n=== {champion_name} Classification Report ===')
print(classification_report(y_test, champion_preds, target_names=['No Subscription', 'Subscription']))

# Visualization suite
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Row 1: Model comparison metrics
ax1 = fig.add_subplot(gs[0, :])
comparison_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].plot(
    kind='bar', ax=ax1, alpha=0.7, edgecolor='black')
ax1.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax1.set_ylabel('Score')
ax1.legend(loc='lower right')
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, 1)
ax1.tick_params(axis='x', rotation=45)

# Row 2: Champion model detailed evaluation
ax2 = fig.add_subplot(gs[1, 0])
ConfusionMatrixDisplay.from_predictions(y_test, champion_preds, ax=ax2, cmap='Blues')
ax2.set_title(f'{champion_name}\nConfusion Matrix', fontweight='bold')

ax3 = fig.add_subplot(gs[1, 1])
RocCurveDisplay.from_predictions(y_test, champion_probs, ax=ax3, color='#3498db', linewidth=2)
ax3.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax3.set_title(f'{champion_name}\nROC Curve', fontweight='bold')
ax3.grid(alpha=0.3)

ax4 = fig.add_subplot(gs[1, 2])
PrecisionRecallDisplay.from_predictions(y_test, champion_probs, ax=ax4, color='#e74c3c', linewidth=2)
ax4.set_title(f'{champion_name}\nPrecision-Recall Curve', fontweight='bold')
ax4.grid(alpha=0.3)

# Row 3: ROC comparison & Feature importance placeholder
ax5 = fig.add_subplot(gs[2, :2])
for model_name, model_obj in models.items():
    probs = model_obj.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax5.plot(fpr, tpr, label=f'{model_name} (AUC={roc_auc_score(y_test, probs):.3f})', linewidth=2)
ax5.plot([0, 1], [0, 1], 'k--', label='Random', alpha=0.3)
ax5.set_xlabel('False Positive Rate')
ax5.set_ylabel('True Positive Rate')
ax5.set_title('ROC Curves - All Models', fontsize=12, fontweight='bold')
ax5.legend(loc='lower right', fontsize=9)
ax5.grid(alpha=0.3)

# Business impact summary
ax6 = fig.add_subplot(gs[2, 2])
ax6.axis('off')
cm = confusion_matrix(y_test, champion_preds)
tn, fp, fn, tp = cm.ravel()
metrics_text = f'''
**Business Impact Summary**

True Positives: {tp:,}
(Correctly predicted subscribers)

False Positives: {fp:,}
(Wasted marketing effort)

False Negatives: {fn:,}
(Missed opportunities)

True Negatives: {tn:,}
(Correctly predicted non-subscribers)

**Key Metrics**
Precision: {precision_score(y_test, champion_preds):.1%}
Recall: {recall_score(y_test, champion_preds):.1%}
F1-Score: {f1_score(y_test, champion_preds):.3f}

**Campaign Efficiency**
Response Rate: {tp/(tp+fp)*100:.1f}%
Coverage: {tp/(tp+fn)*100:.1f}%
'''
ax6.text(0.1, 0.5, metrics_text, fontsize=10, family='monospace', 
         verticalalignment='center')

plt.show()

print('\n✓ Assessment complete')

In [ ]:
from joblib import dump
import json
from datetime import datetime

artifacts = Path('../app/artifacts')
artifacts.mkdir(exist_ok=True, parents=True)

# Save champion model
dump(champion_model, artifacts / 'bank_marketing_semma.joblib')
dump(engineer_features, artifacts / 'feature_engineering_fn.joblib')
print(f'✓ Champion model ({champion_name}) saved to {artifacts / "bank_marketing_semma.joblib"}')
print(f'✓ Feature engineering function saved')

# Create comprehensive model card
model_card = {
    'model_name': 'Bank Marketing Campaign Response Predictor',
    'version': '1.0.0',
    'created_date': datetime.now().isoformat(),
    'methodology': 'SEMMA (Sample, Explore, Modify, Model, Assess)',
    'algorithm': champion_name,
    'training_data': {
        'source': 'Kaggle - Bank Marketing Dataset',
        'full_dataset_size': len(bank),
        'sample_size': len(train_sample),
        'test_size': len(holdout),
        'sampling_strategy': '50% random sample for rapid prototyping',
        'class_distribution': f'{response_counts["no"]:,} no, {response_counts["yes"]:,} yes',
        'imbalance_ratio': f'{response_counts["no"]/response_counts["yes"]:.1f}:1'
    },
    'performance_metrics': {
        'test_accuracy': float(comparison_df[comparison_df['Model']==champion_name]['Accuracy'].values[0]),
        'test_precision': float(comparison_df[comparison_df['Model']==champion_name]['Precision'].values[0]),
        'test_recall': float(comparison_df[comparison_df['Model']==champion_name]['Recall'].values[0]),
        'test_f1': float(comparison_df[comparison_df['Model']==champion_name]['F1-Score'].values[0]),
        'test_roc_auc': float(comparison_df[comparison_df['Model']==champion_name]['ROC-AUC'].values[0]),
        'response_rate': float(tp/(tp+fp)) if (tp+fp) > 0 else 0.0,
        'coverage': float(tp/(tp+fn)) if (tp+fn) > 0 else 0.0
    },
    'semma_phases': {
        'sample': 'Stratified 50% random sample from training data',
        'explore': 'Comprehensive EDA of demographics, job, campaign features',
        'modify': 'Feature engineering: age groups, contact intensity, economic indicators',
        'model': f'Comparison of 5 algorithms, selected {champion_name}',
        'assess': 'ROC/PR curves, confusion matrix, business impact analysis'
    },
    'engineered_features': [
        'age_group: Life stage categorization',
        'contact_intensity: Total campaign + previous contacts',
        'economic_score: Combined economic indicators',
        'has_previous: Binary previous contact flag',
        'prev_success: Previous campaign success indicator'
    ],
    'business_requirements': {
        'primary_metric': 'ROC-AUC for ranking customers',
        'objective': 'Optimize marketing campaign targeting',
        'use_case': 'Predict term deposit subscription likelihood'
    },
    'limitations': [
        'Model trained on 2010-2013 data - may not reflect current behavior',
        'Sample-based training may underfit compared to full dataset',
        'Duration feature has high predictive power but unavailable pre-call',
        'Class imbalance requires threshold tuning for production'
    ],
    'ethical_considerations': [
        'Ensure fair treatment across age and job categories',
        'Avoid over-contacting customers (campaign fatigue)',
        'Provide opt-out mechanisms',
        'Monitor for discriminatory patterns in predictions'
    ],
    'deployment_notes': {
        'preprocessing': 'Apply feature engineering function before scoring',
        'threshold': 0.3,
        'inference_mode': 'Batch scoring for campaign targeting',
        'monitoring': 'Track response rate, precision, and campaign ROI monthly'
    }
}

with open(artifacts / 'model_card.json', 'w') as f:
    json.dump(model_card, f, indent=2)

print(f'✓ Model card saved to {artifacts / "model_card.json"}')
print(f'\n📦 All deployment artifacts ready in {artifacts}/')
print(f'\n=== SEMMA Methodology Complete ===')
print('Sample → Explore → Modify → Model → Assess ✓')

### Next Steps- Integrate AutoGluon for model comparison.- Design uplift modeling experiment.- Prototype SAS ODA replication if license available.